# Phần 10 — Train F01/seed2
Final T04 CutMix recipe + I07.
Train độc lập 12 epoch, batch 64; test tắt trong notebook này.
Dùng thay cho phần training của master; không chạy master đồng thời với notebook riêng.
Một notebook/kernel cho mỗi job; khuyến nghị tuần tự trên RTX 3060.
Master final_10_convnext_tiny.ipynb chỉ mở test sau khi đủ toàn bộ 6 checkpoint/temperature.

## 1. Môi trường

In [ ]:
import os, sys, json
from pathlib import Path
ROOT_HINT = None
ROOT = Path(ROOT_HINT).expanduser().resolve() if ROOT_HINT else next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / "eval.py").is_file() and (p / "starter").is_dir()), None)
if ROOT is None: raise FileNotFoundError("Đặt ROOT_HINT tới gốc repo")
CODE = ROOT / "submissions/2A202602810_NguyenThaiAnh/code"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ.setdefault("MPLCONFIGDIR",str(ROOT / ".cache/matplotlib"))
sys.path.insert(0,str(ROOT)); sys.path.insert(0,str(CODE))
import pandas as pd
from IPython.display import display, Image, Markdown
import final_notebook as fn
import train as trainer
assert Path(trainer.__file__).resolve() == CODE / "train.py"
FINAL_ROOT = fn.root_path()
print("ROOT:", ROOT)
print("Python:", sys.executable)
print("Final output:", FINAL_ROOT)

## 2. Manifest và cấu hình

In [ ]:
ROLE = "F01"
SEED = 2
RUN_TRAINING = True
manifest = fn.load_manifest(FINAL_ROOT) if (FINAL_ROOT / "manifest.json").exists() else (fn.lock_configuration(FINAL_ROOT) if RUN_TRAINING else None)
if manifest is not None:
    entry = fn.entry_for(manifest, ROLE, SEED)
    display(pd.DataFrame([entry["source_cfg"]]).T)
else:
    print("Chưa khóa cấu hình chung kết.")

## 3. Train và fit validation
Chỉ tạo raw training predictions và calibrated val products. Không tạo test loader.
Run đã đủ epoch được audit/đọc lại, run dở resume từ last.pt; config/scheduler giữ nguyên.

In [ ]:
if RUN_TRAINING:
    prepared = fn.train_entry(ROLE, SEED, FINAL_ROOT)
    display(prepared)
else:
    print("Chế độ đọc; không train hoặc đánh giá test.")
display(fn.collect_progress(FINAL_ROOT))

## 4. Curve và bước tiếp theo

In [ ]:
if manifest is not None:
    cfg = trainer.Config(**entry["source_cfg"])
    curve = Path(cfg.curves_dir) / f"{cfg.exp_id}_seed{cfg.seed}_{cfg.backbone}.png"
    if curve.exists(): display(Image(filename=str(curve)))
print("Khi đủ bốn run seed 1/2: mở master phần 10 để khóa checkpoint set và đánh giá test.")